In [1]:
#!pip install rvc3python --upgrade

## Aparcamiento (toma 1)

#### Declaración de funciones

In [ ]:
import numpy as np
from spatialmath.base import plotvol2, transl2, trot2, trplot2, plot_point, trnorm2
import matplotlib.pyplot as plt

from funcionesAuxiliares import transformacion2D, inversa2D, \
                                angulo, perpendicular, transfCoord, \
                                inicializa, muestraMovimiento

### Sistemas de coordenadas

In [ ]:
P=(5,7)

vectorTranslación = (4,4)
T0 = transl2 (vectorTranslación)
coord1 = transfCoord (T=T0, coord=P)

T1 = trot2 (np.pi/3)
coord2 = transfCoord (T=T0@T1, coord=P)

print ('Coordenadas en el sistema inicial: ', P)
print ('Coordenadas en el sistema desplazado ', vectorTranslación,' =  ', coord1.ravel())
print ('Coordenadas en el sistema tras translación y rotación de 60º: ', coord2.ravel())

### Inicialización

In [ ]:
P1, P2, T0 = inicializa()
plot_point (P1, color='red',marker='o')
plot_point (P2, color='red',marker='o')
trplot2 (T0)
print ('Puntos de refrencia:',P1,P2)
print ('Pose inicial:\n',T0)
print ('\t ángulo pose: ', angulo((T0[0,0],T0[0,1])), ' radianes')
print ('\t translación pose: ', T0[:2,2])

### Primera opción: Translación + rotación

In [ ]:
objetivo = (P1+P2)/2

# Translación hasta el centro
coord = transfCoord (coord=objetivo, T=T0)
T1 = transl2(coord)

# Rotación para alinear el coche con los pivotes
v12 = P2-P1
alpha = angulo (v12)
T2 = trot2(alpha+np.pi/2)

plot_point (P1, color='red',marker='o')
plot_point (P2, color='red',marker='o')
trplot2 (T0, color='black')
trplot2 (T0@T1, color='blue')
trplot2 (T0@T1@T2, color='green')

#### El desplazamiento anterior, mostrado como un twist

In [ ]:
plot_point (P1, color='red',marker='o')
plot_point (P2, color='red',marker='o')

muestraMovimiento ((T0,T1@T2))
trplot2 (T0, color='black')
trplot2 (T0@T1@T2, color='green')

### Mi primer twist

In [ ]:
R = 5  #radio (arbitrario)
Periodo = 28 # El periodo de la órbita lunar, en días

P0  = transl2 ((0,-R)) 
v = 2*np.pi*R /Periodo
w = 2*np.pi /Periodo
dT = transl2((v,0)) @ trot2(w)
T = P0
trplot2(P0)
for dia in range(Periodo):
    T=trnorm2(T@dT)
    trplot2(T)

### Segunda opción: algo más sofisticado
Nuestro objetivo es minimizar $(d, |\theta|)$, donde $d$ es la distancia entre el coche y el objetivo (el punto medio entre $\vec{P1}$ y $\vec{P2}$) y $\theta$ el ángulo con la perpendicular a $\vec{P2}-\vec{P1}$

#### i) ¿Cuál es la distancia $d$ entre la pose inicial y el objetivo?

#### ii) Buscamos los dos puntos que están a distancia $d$ del objetivo y son perpendiculares a $\vec{P2}-\vec{P1}$
De ellos, escogemos el más próximo a la pose inicial: lo llamaremos **punto intermedio**

#### iii) Giramos el coche para que esté perpendicular al objetivo, apuntando al **punto intermedio**
Eso nos da nuestra primera transformación: T1

#### iv) El coche se mueve a lo largo de la circunferencia de radio $d$
Este movimiento será un *twist*. Partiendo de la pose T0@T1 debe:

* Desplazarse siguiendo un arco: ¿cuál es la longitud de ese arco?
* Ir girando sobre sí mismo para mantenrse perpendicular al radio de la circunferencia: ¿cuál es el ángulo de giro?

Igual que hicimos con la luna, descomponemos el movimiento en pequeños movimientos *infinitesimales*:
* Define dT: la composición de una translación de longitud *arco*/100 y una rotación de ángulo *anguloGiro*/100
* Compón dT@dT@dT... 100 veces : ese es el twist que buscamos

#### v) Ya estamos en el punto intermedio: ahora giramos el coche para que apunte al objetivo

#### vi) Ya sólo nos queda desplazarnos en línea recta hasta el objetivo...

#### vii)... y representarlo gráficamente
Añade:
* El punto intermedio
* Opcionalemente, las poses intermedias
* Representa el movimiento total usando la función muestraMovimiento

In [ ]:
plot_point (P1, color='red',marker='o')
plot_point (P2, color='red',marker='o')
plot_point ((P1+P2)/2, color='red',marker='x',label='Objetivo')


trplot2 (T0, color='black')

plt.legend(loc='center left', bbox_to_anchor=(1.05, 0.5))